# Group Project Cryosphere

Earth Observation group project about a glacier in the Himalaya in Nepal. We try to measure glacier area, surface temperature variation, and wet/total snow area using satellite data from Google Earth Engine. We investigate these aspects on the time series of 2015 - 2026.

## Initializing

In [1]:
import time
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd
from plotting import *
from gst_series import *
from sar_snow import *
from gst_pixel import *

In [2]:
# run once at beginning, supply token
import ee
ee.Authenticate()

True

In [3]:
GEE_PROJECT = "my-hello-world-508910"  # replace with your own project id
ee.Initialize(project=GEE_PROJECT)

In [4]:
ROI = [[81.768322, 30.198924], [81.785316, 30.200185], [81.782656, 30.20241], [81.780596, 30.2079], [81.780252, 30.215613], [81.780424, 30.223327], [81.778965, 30.228741], [81.776733, 30.231707], [81.774759, 30.231781], [81.771927, 30.225403], [81.77021, 30.216503], [81.770382, 30.211237], [81.770296, 30.206416], [81.766691, 30.202707], [81.765575, 30.199962], [81.768322, 30.198924]]
AOI = ee.Geometry.Polygon(ROI) # from WhatsApp chat
aoi_center_lon_lat = AOI.centroid().coordinates().getInfo() # lon/lat
print(aoi_center_lon_lat)

[81.77534281078347, 30.212620057076144]


In [5]:
START = "2014-09-01"
END = "2026-12-31"

## Image Collections Analysis

Get a sense of how many Images we are working with, from what date they are and how much they are covered by clouds.

In [ ]:
s2_toa = ee.ImageCollection("COPERNICUS/S2_HARMONIZED").filterBounds(AOI).sort("system:time_start").set("collection_name", "Sentinel 2 Top of Atmosphere")
s2_boa = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(AOI).sort("system:time_start").set("collection_name", "Sentinel 2 Bottom of Atmosphere")

In [ ]:
plot_histogram_image_collection(s2_toa, "2015-09-01", "2027-01-01")
plot_histogram_image_collection(s2_boa, "2015-09-01", "2027-01-01")

In these Histograms we see that there Sentinel 2 provides data with few clouds from 2016 to 2026. Also, we see that cloud cover is very seasonal. Especially in summer, images with cloud coverage <30% are scarce. Although more and more images are provided throughout the time period, image count has been very consistent since 2019.

## Glacier area

Description here...

## Surface temperature variation

Description here...

In [6]:
glacier_mask, mask_status = get_glacier_mask(roi=AOI)
print("Glacier mask source:", mask_status)

Glacier mask source: GLIMS/current outlines


In [ ]:
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")
bin_img = dem.divide(BIN_M).floor().multiply(BIN_M).toInt().rename("bin") # 50 bzw. BIN_M grosse Höhenstufen
# show_map(dem=dem, roi=AOI, glacier_mask=glacier_mask) # needs debugging / checking!

In [ ]:
raw_gst = download(bin_img, glacier_mask, AOI, START, END, debug=False)
gst_half_month = process(raw_gst, debug=True)
ablation_trend = ablation_trend(gst_half_month, debug=True)
gst_half_month.head()

In [ ]:
fig_gst_observations_1 = plot_half_months(gst_half_month)

In [ ]:
fig_gst_observations_2 = plot_half_months_2(gst_half_month)

In [ ]:
fig_gst_err = plot_gst_series(gst_half_month, errorbars=True)
fig_gst_noerr = plot_gst_series(gst_half_month, errorbars=False)

In [ ]:
save_any_tables_and_figures(
    {"gst_raw_scene_bins": raw_gst,
     "gst_halfmonthly": gst_half_month,
     "gst_ablation_season": ablation_trend,
     "gst_obervations_1": fig_gst_observations_1,
     "gst_obervations_2": fig_gst_observations_2,
     "gst_err": fig_gst_err,
     "gst_noerr": fig_gst_noerr,
     },
    save_path="results/gst_series/"
)

### GST per pixel

In [ ]:
scenes = download_zone_scenes(AOI, glacier_mask, START, END, debug=True)
half_monthly = process_zones(scenes, debug=True)

winters = winter_means(half_monthly)
print(winter_trends(winters).round(3))

pixel_arrays = image_to_arrays(pixel_statistics(AOI, glacier_mask, START, END), AOI)
fig = plot_pixel_maps(pixel_arrays)
fig.show()

point_series_df = download_point_series({"centre": (81.774, 30.220)}, START, END, debug=True)

save_any_tables_and_figures(
    {"gst_scene_zone_sums": scenes,
     "gst_halfmonthly_winter": half_monthly,
     "gst_winter_means": winters,
     "gst_points": point_series_df,
     "gst_pixel_maps": fig,
     },
    save_path = "results/gst_pixel/"
)

## Wet/total snow area

Description here...

In [ ]:
# can loop over sy and orbit... (but will take a while)
SY = 2024
ORBIT = "ASCENDING"

df, bdf = all_stats(AOI, "2014-09-01", "2026-12-31", glacier_mask=glacier_mask, debug=True)

# wet fraction by elevation band and date
sar_snow_elev = plot_sar_snow_elev(bdf, SY, ORBIT)
# backscatter-change histogram per month, threshold marked
sar_snow_diff_history = show_sar_snow_diff_history(AOI, SY, ORBIT, months=(12, *range(1, 12)))
# monthly wet frequency and melt-onset day
sar_snow_maps = show_sar_snow_maps(AOI, SY, ORBIT, months=(12, *range(1, 12)), glacier_mask=glacier_mask)

outputs = make_outputs(df, bdf)
outputs.update({f"sar_snow_elev_{SY}_{ORBIT}": sar_snow_elev,
                f"sar_snow_diff_history_{SY}_{ORBIT}": sar_snow_diff_history,
                f"sar_snow_maps_{SY}_{ORBIT}": sar_snow_maps})
save_any_tables_and_figures(outputs, "results/sar_snow", debug=True)